## Library

In [5]:
import ccxt.async_support as ccxt
import pandas as pd
import numpy as np
import asyncio
import logging
import os
import dotenv

In [6]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(messages)s')
logger = logging.getLogger(__name__)
dotenv.load_dotenv()

True

In [7]:
API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")

# Variables

In [ ]:
SYMBOL = "ALCHUSD"
WS_DEPTH_URL = "wss://fstream.binance.com/ws/alchusdt@depth10@100ms"
WS_TRADE_URL = "wss://fstream.binance.com/ws/alchusdt@trade"
TIMEFRAME = "1m"

## Exchange Connect

In [9]:
async def create_ccxt_exchange(api_key: str = API_KEY, secret: str = SECRET_KEY):
  exchange = ccxt.binance({
    "apiKey": api_key,
    "secret": secret,
    "enableRateLimit": True,
    "options": {"defaultType": "future"},
  })
  await exchange.load_markets()
  return exchange

## Fetch Data

In [ ]:
async def fetch_data(exchange, symbol: str, timeframe: str, limit: int = 100) -> pd.DataFrame:
  try:
    df = await  exchange.fetch_ohlcv(symbol, timeframe, limit=limit)
    df = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    # ensure numeric types
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.reset_index(drop=True)

  except Exception as e:
    logger.error(f"Error fetching Data: {str(e)}")
    return pd.DataFrame()

## Main Loop

In [ ]:
async def main_loop(exchange, symbol: str):
  v1_history = []
  open_positions = []

  while True:
    try:
      #--------- Fetch MArket Data ----------
      df = await fetch_data(exchange, symbol, TIMEFRAME, limit=500)

    except Exception as e:
      logger.error(f"Main loop error: {str(e)}")
      await asyncio.sleep(0.5)

## Runner

In [ ]:
async def run():
  exchange = await create_ccxt_exchange()
  try:
    await main_loop(exchange,SYMBOL)
  finally:
    await exchange.close()

if __name__ == "__main__":
  try:
    asyncio.run(run())
  except KeyboardInterrupt:
    logger.info("Stopped by user")